In [63]:
# Modeling the Lotka-Volterra (+ diffusion) approach to linguistic evolution
import numpy as np
import math
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import matplotlib.pyplot as plt
import os
import glob
from PIL import Image

In [64]:
# Helpers
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    R = 6371.0
    distance = R * c

    return distance

def solve_de(L, c, u, v, beta, gamma, dt):
    """
    Solves the differential equation, returning (du, dv) to be added to (u, v)
    L: The Laplacian (n x n)
    c: The competition coefficient (positive means u is higher status, negative means v) (n x 1)
    u: The concentrations of Galician speakers (n x 1)
    v: The concentrations of Spanish speakers (n x 1)
    beta: The diffusion coefficients for u and v respectively ((n x 1), (n x 1))
    gamma: The cross-diffusion coefficients for u (n x 1)
    dt: We're solving for du: du/dt --> du = dt(..)
    """

    # Lotka-Volterra Components
    du_lv = (c * u * v) + u * (1-u)
    dv_lv = (-c * u * v) + v * (1-v)

    # Diffusion Component
    du_diff = np.matmul(L, u) * beta[0] + np.matmul(L, (u*v)) * gamma
    dv_diff = np.matmul(L, v) * beta[1] + np.matmul(L, (u*v)) * (1 - gamma)

    # Final Result
    du = dt * (du_lv + du_diff)
    dv = dt * (dv_lv + dv_diff)

    return du, dv

In [65]:
# Read in data
df = pd.read_csv("data/galicia_filtered_census.csv")

In [66]:
# Define initial conditions and coefficients
n = df.shape[0]
u0, v0 = [], []
beta_u = []
beta_v = []
gamma = []
c = []

gamma_const = 1.
c_const = 0.75
beta_u_const = 0.025
beta_v_const = 0.025
high_init_const = (1 + c_const) / (1 + c_const ** 2)
low_init_const = (1 - c_const) / (1 + c_const ** 2)

avg_status = np.mean(df["status"])
randomize_init = True

# Init by status
for _, row in df.iterrows():
    beta_u.append(beta_u_const)
    beta_v.append(beta_v_const)
    curr_high_init_const = np.random.uniform(low_init_const, high_init_const) if randomize_init else high_init_const
    curr_low_init_const = high_init_const + low_init_const - curr_high_init_const if randomize_init else low_init_const
    # Higher status --> more galician
    if row["status"] > avg_status:
        u0.append(curr_high_init_const)
        v0.append(curr_low_init_const)
        c.append(c_const)
        gamma.append(gamma_const)
    else:
        u0.append(curr_low_init_const)
        v0.append(curr_high_init_const)
        c.append(-c_const)
        gamma.append(0)
        
u0 = np.array(u0).reshape(n, 1)
v0 = np.array(v0).reshape(n, 1)
beta_u = np.array(beta_u).reshape(n, 1)
beta_v = np.array(beta_v).reshape(n, 1)
gamma = np.array(gamma).reshape(n, 1)
c = np.array(c).reshape(n, 1)

In [67]:
# Define coords, Laplacian
coords = np.array([df['lat'], df['lon']]).T
populations = np.array(df['pop'])
W = np.zeros((n, n))

# Laplacian at diagonals should be 0...probably.
# Because it represents the weight of diffusion and between a municipality and 
# itself there shouldn't be any diffusion

# NOTE: There is a very big city which is a huge outlier in the population which balloons
# values in the Laplacian.
# Fixes:
# 1. Use log(population) [CURRENT]
#   * Fixes issue, but population weighting is much less severe now, unclear how this will effect simul
# 2. Use normalization between 0, 10000
#   * Should also fix issue, may allow for better edge weighting

for i in range(0, n):
    for j in range(i+1, n):
        weight = (math.log(populations[i], 10) * math.log(populations[j], 10)) / (haversine(coords[i][0], coords[i][1], coords[j][0], coords[j][1]) ** 2)
        W[i][j] = weight
        W[j][i] = weight

DD = np.diag(np.sum(W, axis=1))
laplacian = W - DD

In [68]:
# Simulation
t = 10000
dt = deltat=10.**(-3.)
simul_num = 5
p_strength = 0.1

u_over_time = np.zeros((t, n, 1))
v_over_time = np.zeros((t, n, 1))

for simul_idx in range(simul_num):
    # Perturbation between [-1, 1] normalized to p_strenth of u0/v0
    u_perturbation = p_strength * u0 * (2. * np.random.rand(n, 1)-1)
    v_perturbation = p_strength * v0 * (2. * np.random.rand(n, 1)-1)

    # Starting point for this simulation
    u = np.array(u0 + u_perturbation)
    v = np.array(v0 + v_perturbation)

    for ts in tqdm(range(t), desc=f"Simulation {simul_idx}"):
        du, dv = solve_de(L=laplacian, c=c, u=u, v=v, beta=(beta_u, beta_v),gamma=gamma,dt=dt)
        u, v = u + du, v + dv
        
        u_over_time[ts] += u
        v_over_time[ts] += v

# Get the average values over all simuls
u_over_time = u_over_time / simul_num
v_over_time = v_over_time / simul_num

Simulation 4: 100%|██████████| 10000/10000 [00:02<00:00, 4464.91it/s]


In [69]:
# Wrangle u/v for visualization

# Normalize the frequencies as u[i] = u[i] / (u[i] + v[i])
u_norm = (u_over_time / (u_over_time + v_over_time)).reshape(t, n).round(3)
v_norm = (v_over_time / (u_over_time + v_over_time)).reshape(t, n).round(3)
lang_coeff_over_time = u_norm - v_norm

np.savetxt("data/galician_freq_over_time.csv", u_norm, delimiter=",", fmt="%.3f")
np.savetxt("data/spanish_freq_over_time.csv", v_norm, delimiter=",", fmt="%.3f")
np.savetxt("data/lang_coeff_over_time.csv", lang_coeff_over_time, delimiter=",", fmt="%.3f")

In [70]:
# Visualizing data helpers
def process_frame(gdf, save_path=None):
    # Create a colormap based on the lang_coeff values
    norm = plt.Normalize(vmin=gdf['lang_coeff'].min(), vmax=gdf['lang_coeff'].max())
    cmap = plt.get_cmap('YlOrRd')  # You can use any colormap here

    # Plotting
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_axis_off()

    # Plot each region with color based on its lang_coeff
    gdf.plot(ax=ax, column='lang_coeff', cmap=cmap, linewidth=0.8)
    
    if save_path:
        plt.savefig(save_path, transparent=True)
    plt.close(fig)

In [72]:
# Visualize the language coefficients over each municipality
original_census_df = pd.read_csv("data/galicia_filtered_census.csv")
gdf = gpd.read_file('data/galicia_mun.geojson')
merged_gdf = gdf.merge(original_census_df, left_on='mun_name', right_on='mun', how='left')

# Lang coeff values over time
lang_coeff_df = pd.read_csv("data/lang_coeff_over_time.csv")

save_dir = "output_maps/rand_longer/"

count = 0
for i, row in tqdm(lang_coeff_df.iterrows()):
    if i % 25 == 0:
        save_path = os.path.join(save_dir, f"map_{count}.png")
        merged_gdf["lang_coeff"] = row.to_list()
        process_frame(merged_gdf, save_path)
        count += 1

In [ ]:
# Turn images into gif
def img_to_gif(dir_name, duration=50, extra_at_beg=0, extra_at_end=0):
    frames = []
    num_frames = len(os.listdir(f"output_maps/{dir_name}"))
    for i in range(num_frames):
        curr_frame = [Image.open(image) for image in glob.glob(f"output_maps/{dir_name}/map_{i}.png")][0]
        frames.append(curr_frame)

    for _ in range(extra_at_beg):
        frames.insert(0, frames[0].copy())

    for _ in range(extra_at_end):
        frames.append(frames[-1].copy())

    frame_one = frames[0]
    frame_one.save(f"output_maps/{dir_name}_animated.gif", format="GIF", append_images=frames,
    save_all=True, duration=duration, loop=0)

img_to_gif(save_dir.split("/")[1])